In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
import timm
import matplotlib.pyplot as plt
from tqdm import tqdm

from dataset import prepare_sampled_data, CrackDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [2]:
batch_size = 32
learning_rate = 1e-4
num_epochs = 10
model_name = 'mobilenetv3_large_100'
base_dir = r"D:\Study\HumanStudy\Dataset\Training"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

data_list = prepare_sampled_data(base_dir)

train_size = int(0.8 * len(data_list))
val_size = len(data_list) - train_size
train_list, val_list = torch.utils.data.random_split(data_list, [train_size, val_size])

train_dataset = CrackDataset(train_list, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)

print(f"Train batches: {len(train_loader)}")

[D:\Study\HumanStudy\Dataset\Training]


100%|█████████████████████████████████████████████████████████████████████████| 419995/419995 [41:14<00:00, 169.75it/s]




 전체 데이터 수 : 138,398장
------------------------------
 normal               ( 0) : 20,000장
 crack                ( 1) : 40,000장
 leak                 ( 2) : 15,084장
 efflorescence        ( 3) : 14,920장
 detachment           ( 4) : 13,010장
 reticular crack      ( 5) : 10,806장
 spalling             ( 6) : 10,329장
 material separation  ( 7) : 9,219장
 rebar                ( 8) : 2,139장
 damage               ( 9) : 1,934장
 exhilaration         (10) : 957장
------------------------------
Train batches: 3460


In [3]:
model = timm.create_model(model_name, pretrained=True, num_classes=11)
model = model.to(device)

class MaskedBCELoss(nn.Module):
    def __init__(self, pos_weight=1.0, neg_weight=0.3): 
        super().__init__()
        self.pos_weight = pos_weight
        self.neg_weight = neg_weight 
        self.bce = nn.BCEWithLogitsLoss(reduction='none')

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        
        weight_mask = targets * self.pos_weight + (1 - targets) * self.neg_weight
        
        return (bce_loss * weight_mask).mean()

criterion = MaskedBCELoss(pos_weight=1.0, neg_weight=0.3)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/22.1M [00:00<?, ?B/s]

C:\Users\SehoonChoi\.conda\envs\cuda_default\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SehoonChoi\.cache\huggingface\hub\models--timm--mobilenetv3_large_100.ra_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [4]:
best_loss = float('inf')
history = {'loss': [], 'acc': []}

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    corrects = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
    
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        labels_one_hot = F.one_hot(labels, num_classes=11).float()
        
        outputs = model(images)
        loss = criterion(outputs, labels_one_hot)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()
        
        batch_corrects = (preds * labels_one_hot).sum().item()
        corrects += batch_corrects
        total += labels.size(0)
        
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    epoch_loss = running_loss / total
    epoch_acc = corrects / total
    
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)
    
    print(f"Epoch {epoch+1} - Loss: {epoch_loss:.4f}, Main Label Acc: {epoch_acc:.4f}")
    
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        torch.save(model.state_dict(), f'best_{model_name}_multi_label.pth')
        print(f"Best Model Saved (Loss: {best_loss:.4f})")

Epoch 1/10: 100%|█████████████████████████████████████████████████████| 3460/3460 [18:05<00:00,  3.19it/s, loss=0.0467]


Epoch 1 - Loss: 0.0812, Main Label Acc: 0.7765
Best Model Saved (Loss: 0.0812)


Epoch 2/10: 100%|█████████████████████████████████████████████████████| 3460/3460 [20:25<00:00,  2.82it/s, loss=0.0181]


Epoch 2 - Loss: 0.0520, Main Label Acc: 0.8660
Best Model Saved (Loss: 0.0520)


Epoch 3/10: 100%|█████████████████████████████████████████████████████| 3460/3460 [20:33<00:00,  2.81it/s, loss=0.0158]


...

Epoch 8 - Loss: 0.0144, Main Label Acc: 0.9702
Best Model Saved (Loss: 0.0144)


Epoch 9/10: 100%|█████████████████████████████████████████████████████| 3460/3460 [18:22<00:00,  3.14it/s, loss=0.0176]


Epoch 9 - Loss: 0.0122, Main Label Acc: 0.9754
Best Model Saved (Loss: 0.0122)


Epoch 10/10: 100%|████████████████████████████████████████████████████| 3460/3460 [19:32<00:00,  2.95it/s, loss=0.0157]

Epoch 10 - Loss: 0.0109, Main Label Acc: 0.9783
Best Model Saved (Loss: 0.0109)
